In [ ]:
# 1. Environment Setup
!nvidia-smi
!pip install -q -U torch transformers datasets peft trl bitsandbytes accelerate huggingface_hub

In [ ]:
# 2. Configuration
import os, torch
from huggingface_hub import login

hf_token = input('Enter Hugging Face Write Token: ').strip()
login(token=hf_token)

HF_DATASET_REPO = input('Enter HF Dataset Repo (e.g. user/oss-agent-data): ').strip()
ROLE = 'mason_backend_mason'
BASE_MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
OUTPUT_DIR = f'./trained_adapters/{ROLE}'
HUB_ADAPTER_REPO = f'{HF_DATASET_REPO.split("/")[0]}/oss-agent-{ROLE}-adapter'

In [ ]:
# 3. Ingestion
from datasets import load_dataset
dataset_file = f'datasets/{ROLE}_sft.jsonl'
raw_dataset = load_dataset(HF_DATASET_REPO, data_files={'train': dataset_file}, split='train')
print(f'Loaded {len(raw_dataset)} trajectories.')

In [ ]:
# 4. Model Loading
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

In [ ]:
# 5. QLoRA Training
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)

def apply_chat_template(batch):
    return {'text': [tokenizer.apply_chat_template(c, tokenize=False) for c in batch['messages']]}
dataset = raw_dataset.map(apply_chat_template, batched=True)

args = TrainingArguments(output_dir=OUTPUT_DIR, per_device_train_batch_size=1, gradient_accumulation_steps=4, learning_rate=2e-4, num_train_epochs=3, logging_steps=1, fp16=True, report_to='none')
trainer = SFTTrainer(model=model, train_dataset=dataset, dataset_text_field='text', max_seq_length=2048, tokenizer=tokenizer, args=args)
trainer.train()

In [ ]:
# 6. Upload Weights
model.push_to_hub(HUB_ADAPTER_REPO, token=hf_token, private=True)
tokenizer.push_to_hub(HUB_ADAPTER_REPO, token=hf_token, private=True)
print(f'Adapter weights live at: https://huggingface.co/{HUB_ADAPTER_REPO}')